# PyTorch 5-Fold Training

In [ ]:
# Cell 1: Install and import dependencies
# conda create -n kf_py312 python=3.12
# conda activate kf_py312
# pip install ipykernel uproot h5py numpy torch scikit-learn
## In Ubuntu do:
# sudo apt update
# sudo apt --fix-broken install
# sudo apt install \
#  libsqlite3-0=3.37.2-2ubuntu0.4 \
#  libsqlite3-dev=3.37.2-2ubuntu0.4 \
#  sqlite3=3.37.2-2ubuntu0.4
# sudo apt-mark hold libsqlite3-0 libsqlite3-dev sqlite3
# # OR with conda (not required)
# conda install -c conda-forge openssl sqlite

import sys, os, glob
import numpy as np
import pandas as pd
import h5py
import uproot
import awkward as ak
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("Python", sys.version.split()[0])
print("NumPy ", np.__version__)
print("Pandas", pd.__version__)
print("Awkard", ak.__version__)
print("PyTorch", torch.__version__)

In [ ]:
# Cell 2: Define Data Loader
from sklearn.utils import shuffle as sk_shuffle

# ───────────────────────── 1. Helper functions ──────────────────────────
def load_h5_data(files, label):
    dfs, ys, ws = [], [], []
    for f in files:
        with h5py.File(f, "r") as hf:
            c0 = [n.decode() for n in hf["df/block0_items"][:]]
            v0 = pd.DataFrame(hf["df/block0_values"][:], columns=c0)
            c1 = [n.decode() for n in hf["df/block1_items"][:]]
            v1 = pd.DataFrame(hf["df/block1_values"][:], columns=c1)
        df = pd.concat([v0, v1], axis=1)
        dfs.append(df[selected_variables])
        ws.append(df["weight"].values if "weight" in df.columns else np.ones(len(df)))
        ys.append(np.full(len(df), label))
    return (
        pd.concat(dfs, axis=0).reset_index(drop=True),
        np.concatenate(ys),
        np.concatenate(ws),
    )

def load_root_data(files, label, tree="sel_tree"):
    dfs, ys, ws = [], [], []
    for f in files:
        with uproot.open(f) as rf:
            if tree not in rf: continue
            arr = rf[tree].arrays(selected_variables + ["weight"], library="np")
        dfs.append(pd.DataFrame({v: arr[v] for v in selected_variables}))
        ws.append(arr["weight"]) # Preserving original weight
        ys.append(np.full(arr[selected_variables[0]].shape, label))
    return (
        pd.concat(dfs, axis=0).reset_index(drop=True),
        np.concatenate(ys),
        np.concatenate(ws),
    )

# ───────────────────────── 2. Configuration ─────────────────────────────
selected_variables = [
    "jet1_pt", "jet1_eta", "jet1met_dphi", "jet2met_dphi",
    "met_sig", "mjj",
    "pTjj", "mbb", "pTbb", "dRjj", "dEtajj", "dPhijj", "dRbb", "dEtabb",
    "dPhibb", "jet2_pt", "jet2_eta"
]

base_dir_LQ         = "/home/sgoswami/monobcntuples/"
signal_masses_LQ    = ["500","1000","1400","2000","2500","2800"]
base_dir_stop       = "/home/sgoswami/monobcntuples/run3_btag/all"
signal_pattern_stop = os.path.join(base_dir_stop, "singlestop", "basicSel_sT_*_*.root")

bkg_procs = ["ttbar","singletop","dijet","diboson","wlnu","zll","znunu"]

# ───────────────────────── 3. Discover files ────────────────────────────
def filter_ok(paths):
    return [f for f in paths if "_histogram" not in f and "_cutflow" not in f]

sig_files_LQ = []
for m in signal_masses_LQ:
    sig_files_LQ += filter_ok(glob.glob(os.path.join(base_dir_LQ, f"mass_{m}", "basicSel_mass_*.h5")))
sig_files_stop = filter_ok(glob.glob(signal_pattern_stop))

bkg_files_LQ_flat, bkg_files_stop_flat = [], []
for p in bkg_procs:
    bkg_files_LQ_flat.extend(filter_ok(glob.glob(os.path.join(base_dir_LQ, f"{p}_mc20e", "*.h5"))))
    root_path = os.path.join(base_dir_stop, p, f"basicSel_{p}.root")
    if os.path.exists(root_path):
        bkg_files_stop_flat.append(root_path)

print(">>> DEBUG: File Discovery Complete.")
print(f"    Found {len(sig_files_LQ)} LQ signal files and {len(bkg_files_LQ_flat)} LQ background files.")
print(f"    Found {len(sig_files_stop)} Stop signal files and {len(bkg_files_stop_flat)} Stop background files.")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# NEW LOADING AND BALANCING STRATEGY
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ────────────────── 4. Process LQ Dataset (Signal + Background) ───────────────
print("\n--- Processing LQ Dataset ---")
# 4.1: Load LQ signal
X_LQ_sig, y_LQ_sig, w_LQ_sig = load_h5_data(sig_files_LQ, 1)
N_sig_LQ = len(X_LQ_sig)
print(f"Loaded {N_sig_LQ} LQ signal events. Target for background is {N_sig_LQ}.")

# 4.2: Load all LQ background file-by-file to get their sizes
lq_bkg_dfs, lq_bkg_sizes, lq_bkg_weights = [], [], []
for f in bkg_files_LQ_flat:
    Xf, _, wf = load_h5_data([f], 0)
    lq_bkg_dfs.append(Xf)
    lq_bkg_sizes.append(len(Xf))
    lq_bkg_weights.append(wf)
total_lq_bkg_events = sum(lq_bkg_sizes)
print(f"Found {total_lq_bkg_events} total available LQ background events across {len(lq_bkg_dfs)} files.")

# 4.3: Determine how many events to draw from each LQ background file proportionally
target_per_file_lq = [int(round(N_sig_LQ * (sz / total_lq_bkg_events))) for sz in lq_bkg_sizes]
while sum(target_per_file_lq) < N_sig_LQ: target_per_file_lq[np.argmax(lq_bkg_sizes)] += 1
while sum(target_per_file_lq) > N_sig_LQ: target_per_file_lq[np.argmax(target_per_file_lq)] -= 1

# 4.4: Sample from each LQ background file and shuffle the result
X_LQ_bkg_parts, y_LQ_bkg_parts, w_LQ_bkg_parts = [], [], []
for df, wt, k in zip(lq_bkg_dfs, lq_bkg_weights, target_per_file_lq):
    if k == 0: continue
    sel = np.random.choice(len(df), k, replace=False)
    X_LQ_bkg_parts.append(df.iloc[sel])
    y_LQ_bkg_parts.append(np.zeros(k, dtype=np.int64))
    w_LQ_bkg_parts.append(wt[sel])

X_LQ_bkg = pd.concat(X_LQ_bkg_parts, axis=0)
y_LQ_bkg = np.concatenate(y_LQ_bkg_parts)
w_LQ_bkg = np.concatenate(w_LQ_bkg_parts)

lq_bkg_perm = np.random.permutation(len(X_LQ_bkg))
X_LQ_bkg, y_LQ_bkg, w_LQ_bkg = X_LQ_bkg.iloc[lq_bkg_perm], y_LQ_bkg[lq_bkg_perm], w_LQ_bkg[lq_bkg_perm]
print(f"Sampled and shuffled {len(X_LQ_bkg)} LQ background events.")

# ────────────────── 5. Process Stop Dataset (Signal + Background) ──────────────
print("\n--- Processing Stop Dataset ---")
# 5.1: Load Stop signal
X_stop_sig, y_stop_sig, w_stop_sig = load_root_data(sig_files_stop, 1)
N_sig_stop = len(X_stop_sig)
print(f"Loaded {N_sig_stop} Stop signal events. Target for background is {N_sig_stop}.")

# 5.2: Load all Stop background file-by-file
stop_bkg_dfs, stop_bkg_sizes, stop_bkg_weights = [], [], []
for f in bkg_files_stop_flat:
    Xf, _, wf = load_root_data([f], 0)
    stop_bkg_dfs.append(Xf)
    stop_bkg_sizes.append(len(Xf))
    stop_bkg_weights.append(wf)
total_stop_bkg_events = sum(stop_bkg_sizes)
print(f"Found {total_stop_bkg_events} total available Stop background events across {len(stop_bkg_dfs)} files.")

# 5.3: Determine how many events to draw from each Stop background file proportionally
target_per_file_stop = [int(round(N_sig_stop * (sz / total_stop_bkg_events))) for sz in stop_bkg_sizes]
while sum(target_per_file_stop) < N_sig_stop: target_per_file_stop[np.argmax(stop_bkg_sizes)] += 1
while sum(target_per_file_stop) > N_sig_stop: target_per_file_stop[np.argmax(target_per_file_stop)] -= 1

# 5.4: Sample from each Stop background file and shuffle the result
X_stop_bkg_parts, y_stop_bkg_parts, w_stop_bkg_parts = [], [], []
for df, wt, k in zip(stop_bkg_dfs, stop_bkg_weights, target_per_file_stop):
    if k == 0: continue
    sel = np.random.choice(len(df), k, replace=False)
    X_stop_bkg_parts.append(df.iloc[sel])
    y_stop_bkg_parts.append(np.zeros(k, dtype=np.int64))
    w_stop_bkg_parts.append(wt[sel])

X_stop_bkg = pd.concat(X_stop_bkg_parts, axis=0)
y_stop_bkg = np.concatenate(y_stop_bkg_parts)
w_stop_bkg = np.concatenate(w_stop_bkg_parts)

stop_bkg_perm = np.random.permutation(len(X_stop_bkg))
X_stop_bkg, y_stop_bkg, w_stop_bkg = X_stop_bkg.iloc[stop_bkg_perm], y_stop_bkg[stop_bkg_perm], w_stop_bkg[stop_bkg_perm]
print(f"Sampled and shuffled {len(X_stop_bkg)} Stop background events.")


# ────────────────── 6. Concatenate All Datasets ─────────────────────────
print("\n--- Concatenating and Final Shuffling ---")
X_full = pd.concat([X_LQ_sig, X_LQ_bkg, X_stop_sig, X_stop_bkg], axis=0)
y_full = np.concatenate([y_LQ_sig, y_LQ_bkg, y_stop_sig, y_stop_bkg])
w_full = np.concatenate([w_LQ_sig, w_LQ_bkg, w_stop_sig, w_stop_bkg])
print(f"Concatenated all four datasets. Total events: {len(X_full)}")


# ────────────────── 7. Final Double Shuffle ─────────────────────────────
# First shuffle using numpy permutation
print("Performing first shuffle (numpy)")
perm1 = np.random.permutation(len(X_full))
X_full = X_full.iloc[perm1].reset_index(drop=True)
y_full = y_full[perm1]
w_full = w_full[perm1]

# Second shuffle using scikit-learn's utility (as a different method)
print("Performing second shuffle (sklearn)")
X_full, y_full, w_full = sk_shuffle(X_full, y_full, w_full)
X_full = X_full.reset_index(drop=True)


# ────────────────── 8. Final Output ─────────────────────────────────────
print("\n>>> FINAL DATASET STATS <<<")
print(f"    Total events: {len(X_full)}")
print(f"    Features shape: {X_full.shape}")
print(f"    Final class counts (0=bkg, 1=sig): {np.bincount(y_full.astype(int))}")
print(f"    (Signal should be {N_sig_LQ + N_sig_stop}, Background should be {N_sig_LQ + N_sig_stop})")
# Run this after X_full_df is created
print("Data integrity check:")
print(X_full.describe().transpose())

print("\nChecking for NaN values:")
print(X_full.isna().sum())

In [ ]:
# Cell 3: NN Arch

import torch
import torch.nn as nn

FOLDS = 5
NUM_FEATURES = 5 # Use the number of features as the input dimension

class DNN(nn.Module):
    def __init__(self, in_dim):
        super().__init__()

        self.hidden1 = nn.Linear(in_dim, 128, bias=False)
        self.act1 = nn.GELU()

        self.hidden2 = nn.Linear(128, 64, bias=False)
        self.act2 = nn.GELU()

        self.hidden3 = nn.Linear(64, 32, bias=False)
        self.act3 = nn.GELU()

        self.output_layer = nn.Linear(32, 1, bias=False)
        self.output_act = nn.Sigmoid()

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            # Kaiming/He initialization is still a good choice
            nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='relu')

    def forward(self, x):
        x = self.act1(self.hidden1(x))
        x = self.act2(self.hidden2(x))
        x = self.act3(self.hidden3(x))
        x = self.output_act(self.output_layer(x))
        return x.squeeze(1)

model = DNN(in_dim=NUM_FEATURES)
print(model)

In [ ]:
# Cell 4 - Revised Training Loop

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import numpy as np

lr = 1e-5
epochs = 100
batch_size = 1024
FOLDS = 5

X_arr = X_full.values.astype(np.float32)
y_arr = y_full.astype(np.float32)
w_arr = np.clip(w_full.astype(np.float32), -10, 10)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on {device} with LR={lr}, BS={batch_size}")

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_arr, y_arr), 1):
    print(f"\n=== Fold {fold}/{FOLDS} ===")

    train_losses, val_losses, train_accs, val_accs = [], [], [], []

    X_tr_raw, X_va_raw = X_arr[tr_idx], X_arr[va_idx]
    y_tr, y_va = y_arr[tr_idx], y_arr[va_idx]
    w_tr, w_va = w_arr[tr_idx], w_arr[va_idx]

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr_raw)
    X_va = scaler.transform(X_va_raw)

    train_ds = torch.utils.data.TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr), torch.from_numpy(w_tr))
    val_ds = torch.utils.data.TensorDataset(torch.from_numpy(X_va), torch.from_numpy(y_va), torch.from_numpy(w_va))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    model = DNN(X_tr.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss(reduction="none")

    best_val_loss, patience = float("inf"), 0
    for epoch in range(1, epochs + 1):
        model.train()
        tr_loss_sum, tr_correct, tr_total = 0.0, 0, 0
        for xb, yb, wb in train_loader:
            xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)

            preds = model(xb)
            loss = (criterion(preds, yb) * wb).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            tr_loss_sum += loss.item() * xb.size(0)
            predicted_class = (preds > 0.5).float()
            tr_correct += (predicted_class == yb).sum().item()
            tr_total += yb.size(0)

        train_loss = tr_loss_sum / tr_total
        train_acc = tr_correct / tr_total
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        model.eval()
        va_loss_sum, va_correct, va_total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb, wb in val_loader:
                xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)

                preds = model(xb)
                va_loss_sum  += (criterion(preds, yb) * wb).sum().item()
                predicted_class = (preds > 0.5).float()
                va_correct += (predicted_class == yb).sum().item()
                va_total += yb.size(0)

        val_loss = va_loss_sum / va_total
        val_acc = va_correct / va_total
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        print(f"Epoch {epoch:3d}/{epochs}  "
              f"loss={train_loss:.4f}  acc={train_acc:.4f}  "
              f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}", end="")

        if val_loss < best_val_loss:
            best_val_loss, patience = val_loss, 0
            torch.save(model.state_dict(), f"model_fold{fold}.pt")
            print("  -> new best model saved")
        else:
            patience += 1
            print("")
            if patience >= 20:
                print("   early stopping")
                break

In [ ]:
# Cell 5: Final Evaluation Cell (run after all folds are trained)

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import StandardScaler

print("--- Starting Final K-Fold Evaluation ---")

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

all_true_labels = []
all_pred_probs = []
per_fold_aucs = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_arr, y_arr), 1):
    print(f"Loading and evaluating Fold {fold}...")

    X_tr_raw, X_va_raw = X_arr[tr_idx], X_arr[va_idx]
    y_va = y_arr[va_idx]

    scaler = StandardScaler()
    scaler.fit(X_tr_raw)
    X_va = scaler.transform(X_va_raw)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = DNN(X_va.shape[1]).to(device)
    model.load_state_dict(torch.load(f"model_fold{fold}.pt", map_location=device))
    model.eval()

    with torch.no_grad():
        val_ds = torch.utils.data.TensorDataset(torch.from_numpy(X_va))
        val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size * 2)

        fold_probs = []
        for (xb,) in val_loader:
            probs = model(xb.to(device))
            fold_probs.append(probs.cpu().numpy())

    fold_pred_probs = np.concatenate(fold_probs)
    all_pred_probs.append(fold_pred_probs)
    all_true_labels.append(y_va)

    fold_auc = roc_auc_score(y_va, fold_pred_probs)
    per_fold_aucs.append(fold_auc)
    print(f"  AUC for Fold {fold}: {fold_auc:.4f}")

y_true_full = np.concatenate(all_true_labels)
y_pred_full = np.concatenate(all_pred_probs)

mean_auc = np.mean(per_fold_aucs)
std_auc = np.std(per_fold_aucs)
print(f"\nOverall Model Performance:")
print(f"  Mean AUC = {mean_auc:.4f}")
print(f"  AUC Std. Dev. = {std_auc:.4f}")

fpr, tpr, _ = roc_curve(y_true_full, y_pred_full)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'Overall AUC = {roc_auc:.4f}', lw=2)
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Overall ROC Curve (from all folds)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
plt.hist(y_pred_full[y_true_full==0], bins=50, alpha=0.6, label='Background', density=True)
plt.hist(y_pred_full[y_true_full==1], bins=50, alpha=0.6, label='Signal', density=True)
plt.xlabel('Predicted Probability for Signal Class')
plt.ylabel('Density')
plt.title('Overall Output Distribution (from all folds)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Confusion matrix
# This runs after -Fold evaluation loop has produced y_true_full and y_pred_full

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Convert probabilities to class predictions (0 or 1)
y_pred_class = (y_pred_full > 0.5).astype(int)

cm = confusion_matrix(y_true_full, y_pred_class)
cm_norm = cm.astype(float) / cm.sum(axis=1)[:, None]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Raw counts
disp1 = ConfusionMatrixDisplay(cm, display_labels=['Background','Signal'])
disp1.plot(ax=ax1, values_format='d', cmap='Blues', colorbar=False)
ax1.set_title('Confusion Matrix (counts)')

# Normalized
disp2 = ConfusionMatrixDisplay(cm_norm, display_labels=['Background','Signal'])
disp2.plot(ax=ax2, values_format='.2f', cmap='Blues', colorbar=False)
ax2.set_title('Confusion Matrix (normalized)')

plt.tight_layout()
plt.show()